In [1]:

"""
Inventory Check Against Purchase Report Generator.

This module provides functionality to check and compare inventory movements
against purchase orders, including GRN (Goods Received Note) verification.
"""

from datetime import datetime, timedelta, date
from typing import Optional, Tuple

import pandas as pd
from sqlalchemy import create_engine

import os
import sys
import warnings


# In Jupyter notebooks, __file__ is not defined, so we use getcwd() instead
CURRENT_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# === 2. Import shared modules ===
from mail import send_mail, get_email_recipients
from project_config import DATABASE_URL

# === 3. Configuration ===

engine = create_engine(DATABASE_URL)

# === 5. Suppress warnings ===

warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)
pd.set_option('display.float_format', '{:.2f}'.format)

# Date constants
NOW = datetime.now() 
NOW_STR = NOW.strftime('%Y-%m-%d')

# Example of date constants
# NOW = datetime(2025, 9, 21)
# NOW_STR = '2025-09-21'
YESTERDAY = NOW - timedelta(1)
YESTERDAY_STR = YESTERDAY.strftime('%Y-%m-%d')

🚀 Using .env → Production mode (localhost)


In [4]:
df = pd.read_excel('main.xlsx')
df

,Code,Description,Std_Cost,Last_Pur,Group,Stock_unit,Std_Price,ERP_Stock,PHY_Stock,Sheet_No
0,01-001,6 pcs Precision Screw Driver Set,220.00,220.00,Screwdriver,Pcs,300.00,0.00,NaN,NaN
1,01-002,8 pcs Kex key set SS Audi,55.00,55.00,Screwdriver,Pcs,100.00,8.00,NaN,NaN
2,01-003,1/4 Air compressor key,96.00,96.00,Screwdriver,Pcs,135.00,2.00,NaN,NaN
3,01-004,Air Duster Gun with Short Nozzle DG-10-2,240.00,240.00,Screwdriver,Pcs,336.00,6.00,NaN,NaN
4,01-005,Screw Driver Packet 2pcs (blis) Hmbr,33.00,39.00,Screwdriver,Pcs,44.00,0.00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
15040,41-022,Wheel Brow 1 Wheel,3300.00,3300.00,Trolley,Pcs,4500.00,0.00,NaN,NaN
15041,41-023,Trolly 150KG Steel,3600.00,3600.00,Trolley,Pcs,5750.00,0.00,NaN,NaN
15042,41-026,150kg Hand Trolley Ingco Brand HPHT11502,3315.00,NaN,Trolley,pcs,5304.00,NaN,NaN,NaN
15043,42-002,Cycle Pumper Foot Operated,1260.00,NaN,Pumper,Pcs,1827.00,1.00,NaN,NaN


In [ ]:
import pandas as pd

# ---------- Helper Functions ----------

def normalize_stock(stock_str):
    stock_str = str(stock_str).strip()
    if stock_str == "" or stock_str.lower() == "nan":
        return ""

    cleaned = []
    for v in stock_str.split(","):
        try:
            cleaned.append(str(int(float(v))))
        except ValueError:
            pass

    return ",".join(cleaned)


def calculate_total(stock_str):
    stock_str = str(stock_str).strip()
    if stock_str == "":
        return ""

    total = 0
    for v in stock_str.split(","):
        try:
            total += int(float(v))
        except ValueError:
            pass

    return total


# ---------- Load Excel Files ----------

main_df = pd.read_excel("main.xlsx")
sheet_df = pd.read_excel("sheet.xlsx")   # columns: item_code, qty, sheet_num


# ---------- Ensure Required Columns Exist ----------

for col in ["PHY_Stock", "Sheet_No", "total_sum"]:
    if col not in main_df.columns:
        main_df[col] = ""


# ---------- Clean Existing Data ----------

main_df["PHY_Stock"] = main_df["PHY_Stock"].fillna("").apply(normalize_stock)
main_df["Sheet_No"] = main_df["Sheet_No"].fillna("")


# ---------- Track Not Found Items ----------

not_found_rows = []


# ---------- Append Sheet Data ----------

for _, row in sheet_df.iterrows():
    code = str(row["item_code"]).strip()
    qty = str(int(row["qty"]))
    sheet_no = str(int(row["sheet_num"]))

    match = main_df["Code"] == code

    if match.any():
        idx = main_df[match].index[0]

        # Append PHY_Stock
        current_stock = main_df.at[idx, "PHY_Stock"]
        main_df.at[idx, "PHY_Stock"] = (
            qty if current_stock == "" else f"{current_stock},{qty}"
        )

        # Append Sheet_No
        current_sheet = main_df.at[idx, "Sheet_No"]
        main_df.at[idx, "Sheet_No"] = (
            sheet_no if current_sheet == "" else f"{current_sheet},{sheet_no}"
        )
    else:
        not_found_rows.append(row)


# ---------- Normalize Again (after append) ----------

main_df["PHY_Stock"] = main_df["PHY_Stock"].apply(normalize_stock)


# ---------- Calculate total_sum ----------

main_df["total_sum"] = main_df["PHY_Stock"].apply(calculate_total)


# ---------- Save Files ----------

main_df.to_excel("main_updated.xlsx", index=False)

if not_found_rows:
    not_found_df = pd.DataFrame(not_found_rows)
    not_found_df.to_csv("not_found.csv", index=False)
    print("⚠️ not_found.csv created for missing item codes")
else:
    print("✅ All item codes matched successfully")

print("✅ main_updated.xlsx created successfully")


✅ main_updated.xlsx created successfully
